In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1 — Colab setup: install deps + mount Drive
# ══════════════════════════════════════════════════════════════════════════════
import sys
import numpy as np

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Install pinned dependencies
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "kloppy==3.18.0",
        "pyarrow==22.0.0",
        "pandas==2.2.3",
        "numpy==2.1.3",
        "lxml==5.3.0",
        "scikit-learn==1.6.1",
        "scipy==1.13.1", # Added to ensure compatibility with numpy 2.1.3
    ], check=True)

    # Mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

    base_root = "/content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON"

else:
    print("Not running in Colab — using local paths.")

    # Example local fallback
    base_root = "./Superliga_2024_2025_ALL_SEASON"

# Define paths (works in both environments)
results_path = f"{base_root}/F1_Results_Feed"
squads_path = f"{base_root}/F40_Squads_Feed"
matchfeeds_path = f"{base_root}/Matchfeeds"

print("Paths ready:")
print("Results:", results_path)
print("Squads:", squads_path)
print("Matchfeeds:", matchfeeds_path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Paths ready:
Results: /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/F1_Results_Feed
Squads: /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/F40_Squads_Feed
Matchfeeds: /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/Matchfeeds


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2 — Imports
# ══════════════════════════════════════════════════════════════════════════════
import gc
import json
import time as _time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from lxml import etree
from scipy.spatial import ConvexHull

from kloppy import secondspectrum

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore", category=FutureWarning)

print("Imports ready.")

Imports ready.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3 — Paths (Colab-aware output directories)
# ══════════════════════════════════════════════════════════════════════════════
# We now reuse `base_root` and `matchfeeds_path` defined cleanly in Cell 1!
from pathlib import Path

DATA_ROOT = Path(matchfeeds_path)
OUTPUT_ROOT = Path(base_root)

EXPECTED_FULL_MATCH_COUNT = 192

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {DATA_ROOT}. Please check your Drive mounting.")

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PER_MATCH_DIR = OUTPUT_ROOT / "press_df_v2_per_match"
PER_MATCH_DIR.mkdir(parents=True, exist_ok=True)

FINAL_PARQUET_PATH = OUTPUT_ROOT / "press_df_v2_FINAL.parquet"

print(f"DATA_ROOT       = {DATA_ROOT}")
print(f"PER_MATCH_DIR   = {PER_MATCH_DIR}")
print(f"FINAL_PARQUET   = {FINAL_PARQUET_PATH}")

DATA_ROOT       = /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/Matchfeeds
PER_MATCH_DIR   = /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/press_df_v2_per_match
FINAL_PARQUET   = /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/press_df_v2_FINAL.parquet


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4 — Discover match folders
# ══════════════════════════════════════════════════════════════════════════════
all_match_folders = sorted(
    d for d in DATA_ROOT.iterdir()
    if d.is_dir() and not d.name.startswith(".") and d.name != "ZIP USED"
)

match_folders = sorted(
    d for d in all_match_folders if any(d.glob("*_SecondSpectrum_Data.*"))
)
missing_tracking_folders = [d for d in all_match_folders if d not in match_folders]

print(f"Discovered {len(all_match_folders)} folders, {len(match_folders)} tracking-ready.")
if missing_tracking_folders:
    print("Folders without SecondSpectrum tracking (excluded):")
    for d in missing_tracking_folders:
        print(f"  - {d.name}")

Discovered 193 folders, 192 tracking-ready.
Folders without SecondSpectrum tracking (excluded):
  - 2024-10-06 SønderjyskE - FC Nordsjælland (2442608)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5 — Kloppy tracking loader + Opta parsers
# ══════════════════════════════════════════════════════════════════════════════
def load_tracking_kloppy(match_dir: Path):
    data_files = list(match_dir.glob("*_SecondSpectrum_Data.json*"))
    data_file = [f for f in data_files if f.suffix in (".json", ".jsonl")][0]
    meta_xml = list(match_dir.glob("*_SecondSpectrum_Metadata.xml"))[0]
    meta_json = list(match_dir.glob("*_SecondSpectrum_Metadata.json"))[0]

    dataset = secondspectrum.load(
        raw_data=str(data_file),
        meta_data=str(meta_xml),
        additional_meta_data=str(meta_json),
        coordinates="secondspectrum",
    )
    df = dataset.to_df(
        additional_columns={
            "period_id": lambda f: f.period.id if f.period else None,
            "timestamp_s": lambda f: f.timestamp.total_seconds() if f.timestamp else None,
        }
    )
    return dataset, df


def parse_f24(match_dir: Path) -> pd.DataFrame:
    f24_file = list(match_dir.glob("f24-*.xml"))[0]
    tree = etree.parse(str(f24_file))
    game = tree.getroot().find(".//Game")
    match_id = game.get("id")
    home_team = game.get("home_team_id")
    away_team = game.get("away_team_id")
    rows = []
    for ev in game.findall(".//Event"):
        row = dict(ev.attrib)
        row["match_id"] = match_id
        row["home_team"] = home_team
        row["away_team"] = away_team
        rows.append(row)
    df = pd.DataFrame(rows)
    for col in ["type_id", "period_id", "min", "sec", "outcome",
                "x", "y", "team_id", "player_id"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def parse_f73(match_dir: Path) -> pd.DataFrame:
    f73_file = list(match_dir.glob("f73-*.xml"))[0]
    tree = etree.parse(str(f73_file))
    game = tree.getroot().find(".//Game")
    match_id = game.get("id")
    home_team = game.get("home_team_id")
    away_team = game.get("away_team_id")
    rows = []
    for ev in game.findall(".//Event"):
        row = dict(ev.attrib)
        row["match_id"] = match_id
        row["home_team"] = home_team
        row["away_team"] = away_team
        for q in ev.findall("Q"):
            qid = q.get("qualifier_id")
            val = q.get("value", "1")
            known = {
                "210": "end_x", "211": "end_y",
                "5": "pass_direction",
                "212": "pass_length", "213": "pass_angle",
                "72": "body_part", "214": "pass_speed",
                "170": "error",
            }
            col_name = known.get(qid, f"q_{qid}")
            row[col_name] = val
        rows.append(row)
    df = pd.DataFrame(rows)
    for col in ["type_id", "period_id", "min", "sec", "outcome",
                "x", "y", "team_id", "player_id",
                "sequence_id", "possession_id",
                "end_x", "end_y"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def parse_srml(match_dir: Path) -> dict:
    srml_file = list(match_dir.glob("srml-*.xml"))[0]
    tree = etree.parse(str(srml_file))
    root = tree.getroot()
    ns = {"s": root.nsmap.get(None, "")}
    match_info = root.find(".//s:MatchData", ns) or root.find(".//MatchData")
    result = {"match_date": "", "attendance": "", "teams": []}
    if match_info is not None:
        md = match_info.find("s:MatchInfo", ns) or match_info.find("MatchInfo")
        if md is not None:
            result["match_date"] = md.get("Date", "")
            result["attendance"] = md.get("Attendance", "")
    for team_el in root.findall(".//s:TeamData", ns) or root.findall(".//TeamData"):
        team = {
            "team_ref": team_el.get("TeamRef", ""),
            "side": team_el.get("Side", ""),
            "score": int(team_el.get("Score", 0)),
            "lineup": [],
        }
        for player_el in team_el.findall(".//s:MatchPlayer", ns) or team_el.findall(".//MatchPlayer"):
            team["lineup"].append({
                "player_ref": player_el.get("PlayerRef", ""),
                "position": player_el.get("Position", ""),
                "status": player_el.get("Status", ""),
                "shirt_number": player_el.get("ShirtNumber", ""),
            })
        result["teams"].append(team)
    return result


print("Loaders + parsers defined.")

Loaders + parsers defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6 — Bridge: Kloppy → V2 tracking format + meta builder
# ══════════════════════════════════════════════════════════════════════════════
def _parse_lasttouch_from_jsonl(match_dir: Path) -> dict:
    data_files = list(match_dir.glob("*_SecondSpectrum_Data.json*"))
    data_file = [f for f in data_files if f.suffix in (".json", ".jsonl")][0]
    lt_map = {}
    with open(data_file, "r") as fh:
        for line in fh:
            frame = json.loads(line)
            period = frame.get("period", 1)
            gc = round(frame.get("gameClock", 0.0), 3)
            lt_map[(period, gc)] = frame.get("lastTouch", None)
    return lt_map


def kloppy_tracking_to_v2(dataset, kloppy_df: pd.DataFrame,
                          match_dir: Path = None) -> pd.DataFrame:
    lt_map = _parse_lasttouch_from_jsonl(match_dir) if match_dir else {}
    home_team = dataset.metadata.teams[0]
    away_team = dataset.metadata.teams[1]

    rows = []
    for fr in dataset.records:
        period = fr.period.id if fr.period else 1
        gc = fr.timestamp.total_seconds() if fr.timestamp else 0.0

        if fr.ball_coordinates:
            bx = fr.ball_coordinates.x
            by = fr.ball_coordinates.y
            bz = getattr(fr.ball_coordinates, "z", 0.0) or 0.0
        else:
            bx, by, bz = np.nan, np.nan, np.nan
        ball_spd = getattr(fr.ball_coordinates, "speed", 0.0) if fr.ball_coordinates else 0.0

        home_players, away_players = [], []
        for player, coords in fr.players_coordinates.items():
            if not coords:
                continue
            pdata = fr.players_data.get(player)
            spd = getattr(coords, "speed", None) or getattr(pdata, "speed", 0.0) or 0.0
            entry = {
                "playerId": player.player_id,
                "xyz": [coords.x, coords.y, 0.0],
                "speed": spd,
            }
            if player.team == home_team:
                home_players.append(entry)
            else:
                away_players.append(entry)

        rows.append({
            "period": int(period),
            "frameIdx": fr.frame_id if hasattr(fr, "frame_id") else 0,
            "gameClock": gc,
            "wallClock": 0,
            "live": fr.ball_state.value if fr.ball_state else True,
            "lastTouch": lt_map.get((int(period), round(gc, 3)), None),
            "ball_x": bx, "ball_y": by, "ball_z": bz,
            "ball_spd": ball_spd or 0.0,
            "home_players": home_players,
            "away_players": away_players,
        })
    return pd.DataFrame(rows)


def build_meta_from_kloppy(dataset, match_dir: Path) -> dict:
    home_team = dataset.metadata.teams[0]
    away_team = dataset.metadata.teams[1]
    pitch = dataset.metadata.pitch_dimensions
    pitch_length = pitch.x_dim.max - pitch.x_dim.min if pitch else 105.0
    pitch_width = pitch.y_dim.max - pitch.y_dim.min if pitch else 68.0

    player_lookup = {}
    for side, team in (("home", home_team), ("away", away_team)):
        for p in team.players:
            pos = getattr(p, "starting_position", None) or getattr(p, "position", None)
            player_lookup[p.player_id] = {
                "name": p.name or p.player_id,
                "number": getattr(p, "jersey_no", None),
                "position": str(pos) if pos else "",
                "ssiId": p.player_id,
                "optaId": p.player_id,
                "side": side,
            }
    return {
        "pitchLength": pitch_length,
        "pitchWidth": pitch_width,
        "fps": dataset.metadata.frame_rate or 25,
        "player_lookup": player_lookup,
        "homePlayers": [{"optaId": p.player_id} for p in home_team.players],
        "awayPlayers": [{"optaId": p.player_id} for p in away_team.players],
    }


print("V2 bridge functions defined.")

V2 bridge functions defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7 — Utility functions: gameClock, frame index, attack direction
# ══════════════════════════════════════════════════════════════════════════════
def add_gameclock_opta(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    period_offset = df["period_id"].map({1: 0, 2: 45, 16: 0}).fillna(0).astype(int)
    df["gameClock"] = (df["min"] - period_offset) * 60 + df["sec"]
    df.loc[df["period_id"] == 16, "gameClock"] = np.nan
    return df


def convert_opta_coords_to_meters(df, pitch_length, pitch_width):
    df = df.copy()
    if "x" in df.columns:
        df["x_m"] = df["x"] / 100.0 * pitch_length
    if "y" in df.columns:
        df["y_m"] = df["y"] / 100.0 * pitch_width
    if "end_x" in df.columns:
        df["end_x_m"] = df["end_x"] / 100.0 * pitch_length
    if "end_y" in df.columns:
        df["end_y_m"] = df["end_y"] / 100.0 * pitch_width
    return df


def build_frame_index(tracking_df: pd.DataFrame) -> dict:
    index = {}
    for period in tracking_df["period"].unique():
        subset = tracking_df[tracking_df["period"] == period].sort_values("gameClock").reset_index(drop=True)
        index[period] = (subset["gameClock"].values, subset)
    return index


def find_nearest_frame(frame_index: dict, period: int, gameClock: float, max_dt: float = 0.5):
    if period not in frame_index:
        return None
    gc_arr, subset = frame_index[period]
    idx = np.searchsorted(gc_arr, gameClock)
    best_idx, best_dt = None, float("inf")
    for candidate in [max(0, idx - 1), min(len(gc_arr) - 1, idx)]:
        dt = abs(gc_arr[candidate] - gameClock)
        if dt < best_dt:
            best_dt = dt
            best_idx = candidate
    if best_dt > max_dt:
        return None
    return subset.iloc[best_idx]


def detect_attack_direction(store: dict, match_name: str) -> dict:
    f24 = store["f24"]
    frame_index = store["frame_index"]
    home_team_id = None
    for t in store["srml"]["teams"]:
        if t["side"] == "Home":
            home_team_id = t["team_ref"].lstrip("t")
            break
    if home_team_id is None:
        home_team_id = str(f24.iloc[0].get("team_id"))
    attack_dir = {}
    for period in [1, 2]:
        mask = (
            (f24["period_id"] == period)
            & (f24["x"] > 70)
            & (f24["team_id"].astype(str) == str(home_team_id))
            & (f24["gameClock"].notna())
        )
        events = f24[mask]
        if events.empty:
            events = f24[(f24["period_id"] == period) & (f24["x"] > 70) & (f24["gameClock"].notna())]
        ball_xs = []
        for _, ev in events.head(20).iterrows():
            frame = find_nearest_frame(frame_index, ev["period_id"], ev["gameClock"], max_dt=1.0)
            if frame is not None and not np.isnan(frame["ball_x"]):
                ball_xs.append(frame["ball_x"])
        if ball_xs:
            attack_dir[period] = +1 if np.mean(ball_xs) > 0 else -1
        else:
            attack_dir[period] = -1 if period == 1 else +1
    return attack_dir


print("Utility functions defined.")

Utility functions defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8 — STREAMING REPLACEMENT for original 'all_data' bulk-load
# ══════════════════════════════════════════════════════════════════════════════
# Generator: yields ONE match's store at a time and lets the caller release it.
# Peak RAM stays at ~one match's footprint instead of all 192 at once.
def build_match_store(mf: Path) -> dict:
    """Build the full V2 store for a single match."""
    store = {}
    kloppy_ds, kloppy_df = load_tracking_kloppy(mf)
    store["tracking"] = kloppy_tracking_to_v2(kloppy_ds, kloppy_df, match_dir=mf)
    store["meta"] = build_meta_from_kloppy(kloppy_ds, mf)
    del kloppy_ds, kloppy_df

    store["f24"] = parse_f24(mf)
    store["f73"] = parse_f73(mf)
    store["srml"] = parse_srml(mf)

    pl = store["meta"]["pitchLength"]
    pw = store["meta"]["pitchWidth"]
    store["f24"] = convert_opta_coords_to_meters(add_gameclock_opta(store["f24"]), pl, pw)
    store["f73"] = convert_opta_coords_to_meters(add_gameclock_opta(store["f73"]), pl, pw)

    store["frame_index"] = build_frame_index(store["tracking"])
    store["attack_direction"] = detect_attack_direction(store, mf.name)
    return store


def iter_match_stores(folders):
    """Yield (match_name, store) one match at a time."""
    for mf in folders:
        store = build_match_store(mf)
        yield mf.name, store


print("Streaming generator defined: build_match_store(), iter_match_stores().")

Streaming generator defined: build_match_store(), iter_match_stores().


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9 — Spatial features (B1–B12)
# ══════════════════════════════════════════════════════════════════════════════
def extract_player_xy(player_list):
    coords = []
    for p in player_list:
        xyz = p.get("xyz")
        if xyz is not None and len(xyz) >= 2:
            coords.append([xyz[0], xyz[1]])
    return np.array(coords) if coords else np.empty((0, 2))


def extract_player_speeds(player_list):
    return np.array([p.get("speed", 0.0) for p in player_list if p.get("xyz") is not None])


ALL_SPATIAL_KEYS = [
    "press_centroid_x", "press_centroid_y", "opp_centroid_x", "opp_centroid_y",
    "team_length", "team_width", "shape_aspect_ratio",
    "opp_team_length", "opp_team_width", "opp_shape_aspect_ratio",
    "def_line_height",
    "n_pressers_5m", "n_pressers_10m",
    "dist_nearest_1", "dist_nearest_2", "dist_nearest_3",
    "press_surface_area", "press_stretch_index", "opp_stretch_index",
    "ball_dist_to_sideline", "centroid_dist_to_sideline"
]


def compute_spatial_features(frame, pressing_side, attack_sign, pitch_length, pitch_width=68.0):
    if pressing_side == "home":
        press_players = frame["home_players"]; opp_players = frame["away_players"]
        press_sign = attack_sign
    else:
        press_players = frame["away_players"]; opp_players = frame["home_players"]
        press_sign = -attack_sign

    press_xy = extract_player_xy(press_players)
    opp_xy = extract_player_xy(opp_players)
    press_speeds = extract_player_speeds(press_players)
    ball_xy = np.array([frame["ball_x"], frame["ball_y"]])

    result = {}
    if len(press_xy) < 3:
        return {k: np.nan for k in ALL_SPATIAL_KEYS}

    press_centroid = press_xy.mean(axis=0)
    result["press_centroid_x"] = press_centroid[0]
    result["press_centroid_y"] = press_centroid[1]
    if len(opp_xy) >= 1:
        opp_centroid = opp_xy.mean(axis=0)
        result["opp_centroid_x"] = opp_centroid[0]
        result["opp_centroid_y"] = opp_centroid[1]
    else:
        opp_centroid = None
        result["opp_centroid_x"] = np.nan
        result["opp_centroid_y"] = np.nan

    oriented_x = press_xy[:, 0] * press_sign
    result["team_length"] = oriented_x.max() - oriented_x.min()
    result["team_width"] = press_xy[:, 1].max() - press_xy[:, 1].min()
    result["shape_aspect_ratio"] = (
        result["team_length"] / result["team_width"] if result["team_width"] > 0 else np.nan
    )

    opp_sign = -press_sign
    if len(opp_xy) >= 3:
        opp_oriented_x = opp_xy[:, 0] * opp_sign
        result["opp_team_length"] = opp_oriented_x.max() - opp_oriented_x.min()
        result["opp_team_width"] = opp_xy[:, 1].max() - opp_xy[:, 1].min()
        result["opp_shape_aspect_ratio"] = (
            result["opp_team_length"] / result["opp_team_width"] if result["opp_team_width"] > 0 else np.nan
        )
        opp_dists = np.sqrt(np.sum((opp_xy - opp_centroid) ** 2, axis=1))
        result["opp_stretch_index"] = opp_dists.mean()
    else:
        result["opp_team_length"] = np.nan
        result["opp_team_width"] = np.nan
        result["opp_shape_aspect_ratio"] = np.nan
        result["opp_stretch_index"] = np.nan

    result["def_line_height"] = oriented_x.min() + pitch_length / 2
    dists_to_ball = np.sqrt(np.sum((press_xy - ball_xy) ** 2, axis=1))
    result["n_pressers_5m"] = int((dists_to_ball <= 5.0).sum())
    result["n_pressers_10m"] = int((dists_to_ball <= 10.0).sum())
    sorted_dists = np.sort(dists_to_ball)
    result["dist_nearest_1"] = sorted_dists[0] if len(sorted_dists) > 0 else np.nan
    result["dist_nearest_2"] = sorted_dists[1] if len(sorted_dists) > 1 else np.nan
    result["dist_nearest_3"] = sorted_dists[2] if len(sorted_dists) > 2 else np.nan
    try:
        if len(press_xy) >= 3:
            hull = ConvexHull(press_xy)
            result["press_surface_area"] = hull.volume
        else:
            result["press_surface_area"] = np.nan
    except Exception:
        result["press_surface_area"] = np.nan
    dists_to_centroid = np.sqrt(np.sum((press_xy - press_centroid) ** 2, axis=1))
    result["press_stretch_index"] = dists_to_centroid.mean()

    if not np.isnan(frame["ball_y"]):
        result["ball_dist_to_sideline"] = (pitch_width / 2.0) - abs(frame["ball_y"])
    else:
        result["ball_dist_to_sideline"] = np.nan

    if not np.isnan(result["press_centroid_y"]):
        result["centroid_dist_to_sideline"] = (pitch_width / 2.0) - abs(result["press_centroid_y"])
    else:
        result["centroid_dist_to_sideline"] = np.nan

    return result


print(f"Spatial features defined ({len(ALL_SPATIAL_KEYS)} keys).")

Spatial features defined (21 keys).


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10 — Physical / dynamic features (C1–C5) + pressing_intensity & LANES
# ══════════════════════════════════════════════════════════════════════════════
SPRINT_THRESHOLD = 5.5
T_WINDOW = 1.5
SIGMA = 0.45
TAU_R = 0.7
V_MAX = 8.0
V_ACTIVE = 2.0


def compute_player_velocities(players_now, players_prev, dt):
    if dt <= 0:
        return {}
    prev_positions = {}
    for p in players_prev:
        xyz = p.get("xyz")
        if xyz is not None and len(xyz) >= 2:
            pid = p.get("playerId", p.get("number"))
            prev_positions[pid] = np.array([xyz[0], xyz[1]])
    velocities = {}
    for p in players_now:
        xyz = p.get("xyz")
        if xyz is not None and len(xyz) >= 2:
            pid = p.get("playerId", p.get("number"))
            if pid in prev_positions:
                pos_now = np.array([xyz[0], xyz[1]])
                velocities[pid] = (pos_now - prev_positions[pid]) / dt
    return velocities


def compute_time_to_intercept(def_pos, def_vel, target_pos, v_max, tau_r):
    to_target = target_pos - def_pos
    dist = np.linalg.norm(to_target)
    if dist < 0.01:
        return 0.0
    def_speed = np.linalg.norm(def_vel)
    if def_speed > 0.1:
        direction_to_target = to_target / dist
        cos_angle = np.clip(np.dot(def_vel / def_speed, direction_to_target), -1, 1)
        tau_beta = max(0, (1 - cos_angle) / 2) * 1.0
    else:
        tau_beta = 0.5
    return tau_r + tau_beta + dist / v_max


def compute_intercept_probability(t_intercept, t_window, sigma):
    exponent = np.clip((t_intercept - t_window) / sigma, -20, 20)
    return 1.0 / (1.0 + np.exp(exponent))


def compute_pressing_intensity(frame, frame_prev, pressing_side):
    if pressing_side == "home":
        press_players = frame["home_players"]
    else:
        press_players = frame["away_players"]
    ball_xy = np.array([frame["ball_x"], frame["ball_y"]])
    if np.any(np.isnan(ball_xy)):
        return np.nan
    velocities = {}
    if frame_prev is not None:
        dt = frame["gameClock"] - frame_prev["gameClock"]
        prev_players = frame_prev["home_players"] if pressing_side == "home" else frame_prev["away_players"]
        velocities = compute_player_velocities(press_players, prev_players, dt)
    prob_none = 1.0
    n_active = 0
    for p in press_players:
        xyz = p.get("xyz"); spd = p.get("speed", 0.0)
        if xyz is None or len(xyz) < 2 or spd < V_ACTIVE:
            continue
        n_active += 1
        pos = np.array([xyz[0], xyz[1]])
        pid = p.get("playerId", p.get("number"))
        vel = velocities.get(pid, np.array([0.0, 0.0]))
        t_i = compute_time_to_intercept(pos, vel, ball_xy, V_MAX, TAU_R)
        p_i = compute_intercept_probability(t_i, T_WINDOW, SIGMA)
        prob_none *= (1.0 - p_i)
    if n_active == 0:
        return 0.0
    return 1.0 - prob_none


def closest_point_on_segment(p, a, b):
    ab = b - a
    t = np.dot(p - a, ab) / (np.dot(ab, ab) + 1e-9)
    t = np.clip(t, 0.0, 1.0)
    return a + t * ab


def compute_pressing_intensity_lanes(frame, frame_prev, pressing_side):
    if pressing_side == "home":
        press_players = frame["home_players"]
        opp_players = frame["away_players"]
    else:
        press_players = frame["away_players"]
        opp_players = frame["home_players"]

    ball_xy = np.array([frame["ball_x"], frame["ball_y"]])
    if np.any(np.isnan(ball_xy)):
        return np.nan

    velocities = {}
    if frame_prev is not None:
        dt = frame["gameClock"] - frame_prev["gameClock"]
        prev_players = frame_prev["home_players"] if pressing_side == "home" else frame_prev["away_players"]
        velocities = compute_player_velocities(press_players, prev_players, dt)

    active_pressers = []
    for p in press_players:
        xyz = p.get("xyz"); spd = p.get("speed", 0.0)
        if xyz is None or len(xyz) < 2 or spd < V_ACTIVE:
            continue
        pid = p.get("playerId", p.get("number"))
        vel = velocities.get(pid, np.array([0.0, 0.0]))
        pos = np.array([xyz[0], xyz[1]])
        active_pressers.append((pos, vel))

    if not active_pressers:
        return 0.0

    lane_probs = []
    for opp in opp_players:
        oxyz = opp.get("xyz")
        if oxyz is None or len(oxyz) < 2:
            continue
        opp_pos = np.array([oxyz[0], oxyz[1]])
        dist_to_ball = np.linalg.norm(opp_pos - ball_xy)
        if dist_to_ball < 1.0: # Ignore receiver if too close to the ball
            continue

        prob_none = 1.0
        for (d_pos, d_vel) in active_pressers:
            target = closest_point_on_segment(d_pos, ball_xy, opp_pos)
            t_i = compute_time_to_intercept(d_pos, d_vel, target, V_MAX, TAU_R)
            p_i = compute_intercept_probability(t_i, T_WINDOW, SIGMA)
            prob_none *= (1.0 - p_i)

        lane_probs.append(1.0 - prob_none)

    if not lane_probs:
        return 0.0

    return float(np.mean(lane_probs))


PHYS_KEYS = [
    "ball_speed", "nearest_presser_speed", "n_sprinting_pressers",
    "avg_pressing_team_speed", "closing_speed_nearest", "pressing_intensity",
    "pressing_intensity_lanes"
]


def compute_physical_features(frame, frame_prev, pressing_side):
    press_players = frame["home_players"] if pressing_side == "home" else frame["away_players"]
    press_xy = extract_player_xy(press_players)
    press_speeds = extract_player_speeds(press_players)
    ball_xy = np.array([frame["ball_x"], frame["ball_y"]])
    result = {}
    if frame_prev is not None:
        dt_b = frame["gameClock"] - frame_prev["gameClock"]
        if dt_b > 0 and not np.isnan(frame_prev["ball_x"]):
            dx = frame["ball_x"] - frame_prev["ball_x"]
            dy = frame["ball_y"] - frame_prev["ball_y"]
            result["ball_speed"] = np.sqrt(dx ** 2 + dy ** 2) / dt_b
        else:
            result["ball_speed"] = np.nan
    else:
        result["ball_speed"] = np.nan
    if len(press_xy) < 1:
        for k in ["nearest_presser_speed", "n_sprinting_pressers",
                  "avg_pressing_team_speed", "closing_speed_nearest",
                  "pressing_intensity", "pressing_intensity_lanes"]:
            result[k] = np.nan
        return result
    dists = np.sqrt(np.sum((press_xy - ball_xy) ** 2, axis=1))
    nearest_idx = int(np.argmin(dists))
    result["nearest_presser_speed"] = press_speeds[nearest_idx] if nearest_idx < len(press_speeds) else np.nan
    result["n_sprinting_pressers"] = int((press_speeds > SPRINT_THRESHOLD).sum())
    result["avg_pressing_team_speed"] = press_speeds.mean() if len(press_speeds) > 0 else np.nan
    if frame_prev is not None:
        prev_players = frame_prev["home_players"] if pressing_side == "home" else frame_prev["away_players"]
        prev_xy = extract_player_xy(prev_players)
        prev_ball = np.array([frame_prev["ball_x"], frame_prev["ball_y"]])
        if len(prev_xy) > nearest_idx and not np.any(np.isnan(prev_ball)):
            dist_now = dists[nearest_idx]
            dist_prev = np.sqrt(np.sum((prev_xy[nearest_idx] - prev_ball) ** 2))
            dt = frame["gameClock"] - frame_prev["gameClock"]
            result["closing_speed_nearest"] = (dist_prev - dist_now) / dt if dt > 0 else np.nan
        else:
            result["closing_speed_nearest"] = np.nan
    else:
        result["closing_speed_nearest"] = np.nan

    result["pressing_intensity"] = compute_pressing_intensity(frame, frame_prev, pressing_side)
    result["pressing_intensity_lanes"] = compute_pressing_intensity_lanes(frame, frame_prev, pressing_side)
    return result


print("Physical features defined.")

Physical features defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11 — V2 Phase 1: detect_pressing_frames (frozen thresholds)
# ══════════════════════════════════════════════════════════════════════════════
PRESS_MIN_SPEED = 5.0
PRESS_MAX_ANGLE = 60.0
PRESS_MAX_BALL_Z = 1.0
PRESS_SAMPLE_STEP = 5

PRESS_MIN_RUN_FRAMES = 3
PRESS_MIN_CLOSING = 4.0
PRESS_MIN_RUN_DIST = 5.0

PRESS_EPISODE_MERGE = 3.0
PRESS_OUTCOME_WINDOW = 5.0


def detect_pressing_frames(store, match_name):
    trk = store["tracking"]
    attack_dir = store["attack_direction"]
    home_id, away_id = None, None
    for t in store["srml"]["teams"]:
        tid = t["team_ref"].lstrip("t")
        if t["side"] == "Home":
            home_id = tid
        else:
            away_id = tid
    pressing_frames = []
    for period in [1, 2]:
        sign_home = attack_dir[period]
        period_trk = trk[trk["period"] == period].sort_values("gameClock")
        if len(period_trk) < 2 * PRESS_SAMPLE_STEP:
            continue
        sampled = period_trk.iloc[::PRESS_SAMPLE_STEP].reset_index(drop=True)
        for i in range(1, len(sampled)):
            row_curr = sampled.iloc[i]; row_prev = sampled.iloc[i - 1]
            dt = row_curr["gameClock"] - row_prev["gameClock"]
            if dt <= 0 or dt > 1.0:
                continue
            bx = row_curr["ball_x"]; by = row_curr["ball_y"]; bz = row_curr.get("ball_z", 0)
            if pd.isna(bx) or pd.isna(by):
                continue
            if pd.notna(bz) and bz > PRESS_MAX_BALL_Z:
                continue
            ball = np.array([bx, by]); gc = row_curr["gameClock"]
            last_touch = row_curr.get("lastTouch", None)
            teams = [
                ("home", "home_players", home_id, sign_home),
                ("away", "away_players", away_id, -sign_home),
            ]
            for team_side, player_key, team_id, attack_sign in teams:
                if last_touch == team_side:
                    continue
                if bx * attack_sign <= 0:
                    continue
                players_curr = row_curr.get(player_key, [])
                players_prev = row_prev.get(player_key, [])
                if not isinstance(players_curr, list) or not isinstance(players_prev, list):
                    continue
                prev_pos = {}
                for p in players_prev:
                    pid = str(p.get("optaId") or p.get("playerId") or p.get("number", ""))
                    xyz = p.get("xyz")
                    if pid and xyz and len(xyz) >= 2:
                        prev_pos[pid] = np.array(xyz[:2])
                for p in players_curr:
                    pid = str(p.get("optaId") or p.get("playerId") or p.get("number", ""))
                    xyz = p.get("xyz"); spd = p.get("speed", 0)
                    if not pid or not xyz or len(xyz) < 2:
                        continue
                    if spd < PRESS_MIN_SPEED:
                        continue
                    if pid not in prev_pos:
                        continue
                    pos_curr = np.array(xyz[:2]); pos_prev = prev_pos[pid]
                    movement = pos_curr - pos_prev
                    move_dist = np.linalg.norm(movement)
                    if move_dist < 0.05:
                        continue
                    to_ball = ball - pos_curr
                    dist_to_ball = np.linalg.norm(to_ball)
                    if dist_to_ball < 1.0:
                        continue
                    cos_a = np.dot(movement, to_ball) / (move_dist * dist_to_ball + 1e-9)
                    angle = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))
                    if angle > PRESS_MAX_ANGLE:
                        continue
                    dist_prev = np.linalg.norm(ball - pos_prev)
                    closing = dist_prev - dist_to_ball
                    pressing_frames.append({
                        "match_name": match_name, "period": period, "gameClock": gc,
                        "player_id": pid, "team_id": team_id, "team_side": team_side,
                        "speed": spd, "player_x": pos_curr[0], "player_y": pos_curr[1],
                        "ball_x": bx, "ball_y": by, "ball_z": bz if pd.notna(bz) else 0.0,
                        "dist_to_ball": dist_to_ball, "angle_to_ball": angle, "closing": closing,
                    })
    return pressing_frames


print("Phase 1 function defined.")

Phase 1 function defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12 — V2 Phase 2: group_pressing_runs + cluster_runs_into_episodes
# ══════════════════════════════════════════════════════════════════════════════
def _emit_run(runs, grp, start, end):
    seg = grp.iloc[start:end]
    if len(seg) < 2:
        return
    gc_start = seg["gameClock"].iloc[0]; gc_end = seg["gameClock"].iloc[-1]
    coords = seg[["player_x", "player_y"]].values
    deltas = np.diff(coords, axis=0)
    runs.append({
        "match_name": seg["match_name"].iloc[0], "period": seg["period"].iloc[0],
        "player_id": seg["player_id"].iloc[0], "team_id": seg["team_id"].iloc[0],
        "team_side": seg["team_side"].iloc[0],
        "gc_start": gc_start, "gc_end": gc_end, "duration": gc_end - gc_start,
        "n_frames": len(seg),
        "total_dist": float(np.sum(np.linalg.norm(deltas, axis=1))),
        "total_closing": float(seg["closing"].sum()),
        "mean_speed": seg["speed"].mean(), "max_speed": seg["speed"].max(),
        "mean_angle": seg["angle_to_ball"].mean(),
        "start_dist": seg["dist_to_ball"].iloc[0], "end_dist": seg["dist_to_ball"].iloc[-1],
        "ball_x_start": seg["ball_x"].iloc[0], "ball_y_start": seg["ball_y"].iloc[0],
        "ball_x_end": seg["ball_x"].iloc[-1], "ball_y_end": seg["ball_y"].iloc[-1],
        "player_x_start": seg["player_x"].iloc[0], "player_y_start": seg["player_y"].iloc[0],
    })


def group_pressing_runs(press_frames_df):
    MAX_GAP = 2.0 * (PRESS_SAMPLE_STEP / 25.0) + 0.01
    runs = []
    for (match, period, pid), grp in press_frames_df.groupby(["match_name", "period", "player_id"]):
        grp = grp.sort_values("gameClock")
        gcs = grp["gameClock"].values
        seg_start = 0
        for i in range(1, len(gcs)):
            if gcs[i] - gcs[i - 1] > MAX_GAP:
                _emit_run(runs, grp, seg_start, i)
                seg_start = i
        _emit_run(runs, grp, seg_start, len(gcs))
    runs_df = pd.DataFrame(runs)
    if runs_df.empty:
        return runs_df
    runs_df = runs_df[runs_df["n_frames"] >= PRESS_MIN_RUN_FRAMES]
    runs_df = runs_df[runs_df["total_closing"] >= PRESS_MIN_CLOSING]
    runs_df = runs_df[runs_df["total_dist"] >= PRESS_MIN_RUN_DIST]
    return runs_df.reset_index(drop=True)


def _summarise_episode(ep_id, match, period, team_side, runs_list):
    runs = (pd.DataFrame(runs_list)
            if not isinstance(runs_list[0], pd.Series)
            else pd.DataFrame([r.to_dict() for r in runs_list]))
    return {
        "episode_id": ep_id, "match_name": match, "period": period,
        "team_side": team_side, "team_id": runs["team_id"].iloc[0],
        "gc_start": runs["gc_start"].min(), "gc_end": runs["gc_end"].max(),
        "duration": runs["gc_end"].max() - runs["gc_start"].min(),
        "n_runs": len(runs), "n_pressers": runs["player_id"].nunique(),
        "player_ids": list(runs["player_id"].unique()),
        "mean_speed": runs["mean_speed"].mean(), "max_speed": runs["max_speed"].max(),
        "total_closing": runs["total_closing"].sum(), "mean_angle": runs["mean_angle"].mean(),
        "ball_x_start": runs["ball_x_start"].iloc[0], "ball_y_start": runs["ball_y_start"].iloc[0],
        "ball_x_end": runs.loc[runs["gc_end"].idxmax(), "ball_x_end"],
        "ball_y_end": runs.loc[runs["gc_end"].idxmax(), "ball_y_end"],
        "start_dist_min": runs["start_dist"].min(), "end_dist_min": runs["end_dist"].min(),
    }


def cluster_runs_into_episodes(runs_df, ep_id_offset=0):
    if runs_df.empty:
        return pd.DataFrame()
    episodes = []
    ep_id = ep_id_offset
    for (match, period, team_side), grp in runs_df.groupby(["match_name", "period", "team_side"]):
        grp = grp.sort_values("gc_start")
        current_start = current_end = None
        current_runs = []
        for _, run in grp.iterrows():
            if current_start is None:
                current_start = run["gc_start"]; current_end = run["gc_end"]; current_runs = [run]
            elif run["gc_start"] <= current_end + PRESS_EPISODE_MERGE:
                current_end = max(current_end, run["gc_end"]); current_runs.append(run)
            else:
                episodes.append(_summarise_episode(ep_id, match, period, team_side, current_runs))
                ep_id += 1
                current_start = run["gc_start"]; current_end = run["gc_end"]; current_runs = [run]
        if current_runs:
            episodes.append(_summarise_episode(ep_id, match, period, team_side, current_runs))
            ep_id += 1
    return pd.DataFrame(episodes)


print("Phase 2 functions defined.")

Phase 2 functions defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13 — V2 Phase 3: outcome labelling
# ══════════════════════════════════════════════════════════════════════════════
CHANCE_TYPE_IDS = {13, 14, 15, 16}
CLEARANCE_TYPE_ID = 50
SET_PIECE_IDS = {2, 5, 6, 27}
EXCLUDE_CONTEXTS = {6, 2}
SUSTAINED_PASSES = 3
SUSTAINED_SECONDS = 5.0
CHANCE_WINDOW = 10.0


def _find_possession_change(f73_period, pressing_team, gc_start, gc_end, outcome_window):
    window_end = gc_end + outcome_window
    mask = (f73_period["gameClock"] >= gc_start - 0.5) & (f73_period["gameClock"] <= window_end)
    window = f73_period[mask]
    if len(window) < 2:
        return None, None
    poss_ids = window["possession_id"].values
    teams = window["team_id"].astype(str).values
    gcs = window["gameClock"].values
    for i in range(1, len(poss_ids)):
        pid_curr, pid_prev = poss_ids[i], poss_ids[i - 1]
        if pd.isna(pid_curr) or pd.isna(pid_prev):
            continue
        if pid_curr != pid_prev and teams[i] != teams[i - 1]:
            if teams[i] == str(pressing_team):
                return window.index[i], gcs[i]
    return None, None


def _check_sustained_v2(f73_period, regain_idx, pressing_team):
    events_after = f73_period.loc[regain_idx:]
    pass_count = 0
    gc_start = events_after["gameClock"].iloc[0]
    for _, ev in events_after.iloc[1:].iterrows():
        if str(ev["team_id"]) != str(pressing_team):
            break
        if ev["type_id"] == 1:
            pass_count += 1
        duration = ev["gameClock"] - gc_start
        if pass_count >= SUSTAINED_PASSES or duration >= SUSTAINED_SECONDS:
            return True
    return False


def _check_chance_v2(f73_period, regain_idx, pressing_team):
    events_after = f73_period.loc[regain_idx:]
    gc_start = events_after["gameClock"].iloc[0]
    for _, ev in events_after.iterrows():
        if str(ev["team_id"]) != str(pressing_team):
            break
        if ev["gameClock"] - gc_start > CHANCE_WINDOW:
            break
        if ev["type_id"] in CHANCE_TYPE_IDS:
            return True
    return False


def _check_forced_clearance_v2(f73_period, gc_start, gc_end, opponent_team):
    window_end = gc_end + PRESS_OUTCOME_WINDOW
    mask = ((f73_period["gameClock"] >= gc_start) & (f73_period["gameClock"] <= window_end)
            & (f73_period["team_id"].astype(str) == str(opponent_team)))
    for _, ev in f73_period[mask].iterrows():
        if int(ev["type_id"]) == CLEARANCE_TYPE_ID:
            return True
    return False


def _get_trigger_context_v2(f73_period, gc_start):
    mask = (f73_period["gameClock"] >= gc_start - 3.0) & (f73_period["gameClock"] <= gc_start + 1.0)
    for _, ev in f73_period[mask].iterrows():
        tid = int(ev["type_id"])
        if tid in SET_PIECE_IDS:
            if tid in EXCLUDE_CONTEXTS:
                return "set_piece_excluded"
            elif tid == 5:
                return "goal_kick"
            elif tid == 27:
                return "throw_in"
    return "open_play"


def label_episode_outcomes_for_match(episodes_df, store):
    """Per-match version: takes one match's store directly."""
    if episodes_df.empty:
        return episodes_df
    f73 = store["f73"]
    labels = {k: [] for k in [
        "press_start", "press_success_sustained", "press_to_chance",
        "press_forced_clearance", "press_no_effect", "trigger_context",
    ]}
    for _, ep in episodes_df.iterrows():
        period = ep["period"]; pressing_team = str(ep["team_id"])
        teams_in_match = f73["team_id"].dropna().unique().astype(str)
        opponent_team = [t for t in teams_in_match if t != pressing_team]
        opponent_team = opponent_team[0] if opponent_team else None
        f73_p = f73[(f73["period_id"] == period) & (f73["gameClock"].notna())].sort_values("gameClock")
        if len(f73_p) < 2:
            for k in labels:
                labels[k].append(0 if k != "trigger_context" else "unknown")
            continue
        labels["trigger_context"].append(_get_trigger_context_v2(f73_p, ep["gc_start"]))
        change_idx, _ = _find_possession_change(
            f73_p, pressing_team, ep["gc_start"], ep["gc_end"], PRESS_OUTCOME_WINDOW
        )
        if change_idx is not None:
            labels["press_start"].append(1)
            labels["press_success_sustained"].append(int(_check_sustained_v2(f73_p, change_idx, pressing_team)))
            labels["press_to_chance"].append(int(_check_chance_v2(f73_p, change_idx, pressing_team)))
        else:
            labels["press_start"].append(0)
            labels["press_success_sustained"].append(0)
            labels["press_to_chance"].append(0)
        if opponent_team:
            labels["press_forced_clearance"].append(
                int(_check_forced_clearance_v2(f73_p, ep["gc_start"], ep["gc_end"], opponent_team))
            )
        else:
            labels["press_forced_clearance"].append(0)
        if change_idx is None and labels["press_forced_clearance"][-1] == 0:
            labels["press_no_effect"].append(1)
        else:
            labels["press_no_effect"].append(0)
    out = episodes_df.copy()
    for k, v in labels.items():
        out[k] = v
    return out


print("Phase 3 function defined.")

Phase 3 function defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 14 — V2 Phase 4: build per-match press_df_v2
# ══════════════════════════════════════════════════════════════════════════════
EPISODE_COLS = [
    "episode_id", "match_name", "period", "team_side", "team_id",
    "gc_start", "gc_end", "duration", "n_runs", "n_pressers",
    "player_ids", "mean_speed", "max_speed", "total_closing", "mean_angle",
    "ball_x_start", "ball_y_start", "ball_x_end", "ball_y_end",
    "start_dist_min", "end_dist_min",
]
LABEL_COLS_V2 = [
    "press_start", "press_success_sustained", "press_to_chance",
    "press_forced_clearance", "press_no_effect", "trigger_context",
]


def build_press_df_for_match(episodes_df_v2, store):
    """Compute spatial + physical features for one match's episodes; return final DataFrame."""
    if episodes_df_v2.empty:
        return pd.DataFrame()
    fi = store["frame_index"]
    pitch_length = store["meta"]["pitchLength"]
    pitch_width = store["meta"]["pitchWidth"]
    spatial_rows = []; phys_rows = []
    for _, ep in episodes_df_v2.iterrows():
        period = ep["period"]; gc = ep["gc_start"]
        frame = find_nearest_frame(fi, period, gc)
        if frame is None:
            spatial_rows.append({k: np.nan for k in ALL_SPATIAL_KEYS})
            phys_rows.append({k: np.nan for k in PHYS_KEYS})
            continue
        attack_sign = store["attack_direction"][period]
        spatial_rows.append(compute_spatial_features(frame, ep["team_side"], attack_sign, pitch_length, pitch_width))
        frame_prev = find_nearest_frame(fi, period, gc - 0.12, max_dt=0.2)
        phys_rows.append(compute_physical_features(frame, frame_prev, ep["team_side"]))
    spatial_df = pd.DataFrame(spatial_rows)
    phys_df = pd.DataFrame(phys_rows)
    out = pd.concat([
        episodes_df_v2[EPISODE_COLS].reset_index(drop=True),
        episodes_df_v2[LABEL_COLS_V2].reset_index(drop=True),
        spatial_df.reset_index(drop=True),
        phys_df.reset_index(drop=True),
    ], axis=1)
    out["ball_zone_third"] = pd.cut(
        out["ball_x_start"].abs(),
        bins=[0, 52.5 / 3, 2 * 52.5 / 3, 52.5],
        labels=["defensive", "middle", "attacking"],
        include_lowest=True,
    )
    out["minute"] = (out["gc_start"] / 60).round(0).astype(int)
    out.loc[out["period"] == 2, "minute"] += 45
    # Player_ids list -> JSON string for parquet compatibility
    out["player_ids"] = out["player_ids"].apply(lambda v: json.dumps(list(v)) if isinstance(v, (list, tuple)) else v)
    return out


print("Phase 4 builder defined.")

Phase 4 builder defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 15 — Per-match driver: stream all 192 matches → per-match parquet
# ══════════════════════════════════════════════════════════════════════════════
# Resumable: skips matches whose parquet already exists in PER_MATCH_DIR.
#FORCE_REPROCESS = True # Set to True to recalculate everything with new features

t0 = _time.time()
n_done = n_skipped = n_empty = n_failed = 0
failures = []

for i, mf in enumerate(match_folders, 1):
    out_path = PER_MATCH_DIR / f"{mf.name}.parquet"
    if out_path.exists() and not FORCE_REPROCESS:
        n_skipped += 1
        print(f"[{i:>3}/{len(match_folders)}] SKIP (exists): {mf.name}")
        continue

    t_match = _time.time()
    try:
        store = build_match_store(mf)

        # Phase 1
        press_frames = pd.DataFrame(detect_pressing_frames(store, mf.name))
        if press_frames.empty:
            pd.DataFrame().to_parquet(out_path)
            n_empty += 1
            print(f"[{i:>3}/{len(match_folders)}] EMPTY (no press frames): {mf.name}")
            continue

        # Phase 2
        runs = group_pressing_runs(press_frames)
        episodes = cluster_runs_into_episodes(runs)
        if episodes.empty:
            pd.DataFrame().to_parquet(out_path)
            n_empty += 1
            print(f"[{i:>3}/{len(match_folders)}] EMPTY (no episodes): {mf.name}")
            continue

        # Phase 3
        episodes_lab = label_episode_outcomes_for_match(episodes, store)
        episodes_v2 = episodes_lab[episodes_lab["trigger_context"] != "set_piece_excluded"].copy()
        if episodes_v2.empty:
            pd.DataFrame().to_parquet(out_path)
            n_empty += 1
            print(f"[{i:>3}/{len(match_folders)}] EMPTY (all set-piece): {mf.name}")
            continue

        # Phase 4
        match_press_df = build_press_df_for_match(episodes_v2, store)
        match_press_df.to_parquet(out_path, index=False)
        n_done += 1
        dt = _time.time() - t_match
        print(f"[{i:>3}/{len(match_folders)}] OK   {len(match_press_df):4d} eps  [{dt:5.1f}s]  {mf.name}")

    except Exception as e:
        n_failed += 1
        failures.append((mf.name, repr(e)))
        print(f"[{i:>3}/{len(match_folders)}] FAIL: {mf.name}  ->  {e!r}")
    finally:
        # Aggressively free this match's memory before the next one
        for var in ("store", "press_frames", "runs", "episodes",
                    "episodes_lab", "episodes_v2", "match_press_df"):
            if var in dir():
                try:
                    del locals()[var]
                except Exception:
                    pass
        gc.collect()

elapsed = _time.time() - t0
print("\n" + "=" * 70)
print(f"Per-match driver complete in {elapsed/60:.1f} min")
print(f"  ok       : {n_done}")
print(f"  skipped  : {n_skipped} (already had parquet)")
print(f"  empty    : {n_empty} (no episodes)")
print(f"  failed   : {n_failed}")
if failures:
    print("\nFailures:")
    for name, err in failures:
        print(f"  - {name}: {err}")

[  1/192] OK    107 eps  [ 58.5s]  2024-07-19 AGF - FC Midtjylland (2442545)
[  2/192] OK    159 eps  [ 60.0s]  2024-07-19 FC Nordsjælland - AaB (2442546)
[  3/192] OK    104 eps  [ 62.9s]  2024-07-21 Silkeborg IF - SønderjyskE (2442547)
[  4/192] OK     77 eps  [ 68.0s]  2024-07-21 Vejle Boldklub - Randers FC (2442548)
[  5/192] OK    128 eps  [ 66.6s]  2024-07-21 Viborg FF - Brøndby IF (2442549)
[  6/192] OK    114 eps  [ 67.4s]  2024-07-22 Lyngby Boldklub - F.C. København (2442550)
[  7/192] OK    106 eps  [ 64.0s]  2024-07-26 SønderjyskE - Lyngby Boldklub (2442551)
[  8/192] OK    143 eps  [ 64.5s]  2024-07-27 FC Nordsjælland - FC Midtjylland (2442552)
[  9/192] OK    128 eps  [ 66.5s]  2024-07-28 AaB - Silkeborg IF (2442554)
[ 10/192] OK    122 eps  [ 64.7s]  2024-07-28 F.C. København - AGF (2442555)
[ 11/192] OK     89 eps  [ 75.9s]  2024-07-28 Randers FC - Viborg FF (2442553)
[ 12/192] OK     98 eps  [ 70.9s]  2024-07-29 Brøndby IF - Vejle Boldklub (2442556)
[ 13/192] OK    151 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 16 — Streaming concat → press_df_v2_FINAL.parquet (Step 7 / A4)
# ══════════════════════════════════════════════════════════════════════════════
files = sorted(PER_MATCH_DIR.glob("*.parquet"))
print(f"Per-match parquet files found: {len(files)}")

writer = None
total_rows = 0
matches_with_rows = 0
matches_empty = 0

for f in files:
    try:
        t = pq.read_table(f)
    except Exception as e:
        print(f"  skip unreadable {f.name}: {e!r}")
        continue
    if t.num_rows == 0:
        matches_empty += 1
        continue
    if writer is None:
        writer = pq.ParquetWriter(FINAL_PARQUET_PATH, t.schema)
        ref_schema = t.schema
    else:
        # Align to first non-empty schema (safe in our case: all matches share columns)
        if t.schema != ref_schema:
            t = t.cast(ref_schema, safe=False)
    writer.write_table(t)
    total_rows += t.num_rows
    matches_with_rows += 1

if writer is not None:
    writer.close()

print("\n" + "=" * 70)
print("A4 complete — immutable dataset exported")
print("=" * 70)
print(f"Path:                    {FINAL_PARQUET_PATH}")
print(f"Total rows:              {total_rows:,}")
print(f"Matches with rows:       {matches_with_rows}")
print(f"Matches empty (0 eps):   {matches_empty}")
print(f"Total per-match parquet: {len(files)} (expected {EXPECTED_FULL_MATCH_COUNT})")
print("\nFrozen parameters:")
print({
    "PRESS_MIN_SPEED": PRESS_MIN_SPEED,
    "PRESS_MAX_ANGLE": PRESS_MAX_ANGLE,
    "PRESS_MIN_CLOSING": PRESS_MIN_CLOSING,
    "PRESS_MIN_RUN_DIST": PRESS_MIN_RUN_DIST,
    "PRESS_OUTCOME_WINDOW": PRESS_OUTCOME_WINDOW,
})

Per-match parquet files found: 192

A4 complete — immutable dataset exported
Path:                    /content/drive/MyDrive/Superliga_2024_2025_ALL_SEASON/press_df_v2_FINAL.parquet
Total rows:              22,873
Matches with rows:       192
Matches empty (0 eps):   0
Total per-match parquet: 192 (expected 192)

Frozen parameters:
{'PRESS_MIN_SPEED': 5.0, 'PRESS_MAX_ANGLE': 60.0, 'PRESS_MIN_CLOSING': 4.0, 'PRESS_MIN_RUN_DIST': 5.0, 'PRESS_OUTCOME_WINDOW': 5.0}


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 17 — Quick sanity check on the final parquet
# ══════════════════════════════════════════════════════════════════════════════
press_df_FINAL = pd.read_parquet(FINAL_PARQUET_PATH)
print(f"Shape:    {press_df_FINAL.shape}")
print(f"Matches:  {press_df_FINAL['match_name'].nunique()}")
print("\nLabel sums:")
for lbl in ["press_start", "press_success_sustained", "press_to_chance",
            "press_forced_clearance", "press_no_effect"]:
    print(f"  {lbl:30s}: {int(press_df_FINAL[lbl].sum()):,}")
press_df_FINAL.head()

Shape:    (22873, 57)
Matches:  192

Label sums:
  press_start                   : 3,726
  press_success_sustained       : 1,619
  press_to_chance               : 78
  press_forced_clearance        : 660
  press_no_effect               : 18,805


,episode_id,match_name,period,team_side,team_id,gc_start,gc_end,duration,n_runs,n_pressers,player_ids,mean_speed,max_speed,total_closing,mean_angle,ball_x_start,ball_y_start,ball_x_end,ball_y_end,start_dist_min,end_dist_min,press_start,press_success_sustained,press_to_chance,press_forced_clearance,press_no_effect,trigger_context,press_centroid_x,press_centroid_y,opp_centroid_x,opp_centroid_y,team_length,team_width,shape_aspect_ratio,opp_team_length,opp_team_width,opp_shape_aspect_ratio,opp_stretch_index,def_line_height,n_pressers_5m,n_pressers_10m,dist_nearest_1,dist_nearest_2,dist_nearest_3,press_surface_area,press_stretch_index,ball_dist_to_sideline,centroid_dist_to_sideline,ball_speed,nearest_presser_speed,n_sprinting_pressers,avg_pressing_team_speed,closing_speed_nearest,pressing_intensity,pressing_intensity_lanes,ball_zone_third,minute
0,0,2024-07-19 AGF - FC Midtjylland (2442545),1,away,1000,133.8,136.0,2.2,1,1,"[""494784""]",6.328333,6.88,14.538708,17.401499,31.16,-32.96,28.34,-23.86,23.032833,1.052236,0,0,0,0,1,open_play,14.313636,-11.722727,27.770000,-11.577273,68.24,37.85,1.802906,46.84,39.82,1.176293,15.679198,21.20,1,1,2.075307,12.035352,22.051594,1465.63940,20.055055,0.94,22.177273,2.333333,1.74,0,2.005455,-0.073237,0.179969,0.721206,middle,2
1,1,2024-07-19 AGF - FC Midtjylland (2442545),1,away,1000,147.6,150.6,3.0,2,2,"[""477672"", ""551144""]",5.895278,6.83,22.022894,28.317469,42.83,12.09,25.11,22.01,13.990976,2.608371,1,0,0,0,0,open_play,9.994545,-1.317273,20.083636,-0.265455,65.78,40.35,1.630235,46.04,61.55,0.748010,21.804551,18.52,0,0,13.990976,16.537696,24.084653,1623.65890,19.310792,21.81,32.582727,15.624056,5.89,1,3.635455,7.710867,0.123685,0.648669,attacking,2
2,2,2024-07-19 AGF - FC Midtjylland (2442545),1,away,1000,154.8,157.0,2.2,3,3,"[""551144"", ""420965"", ""494784""]",5.693889,6.63,22.311646,27.049080,17.75,1.47,0.43,-17.35,4.415529,15.198487,0,0,0,0,1,open_play,8.337273,8.682727,19.563636,5.684545,64.40,40.95,1.572650,43.76,50.63,0.864310,15.019748,17.03,1,2,4.415529,9.990320,10.963453,1649.72280,17.862450,32.43,25.217273,2.713137,5.05,0,3.124545,1.070647,0.712721,0.790793,middle,3
3,3,2024-07-19 AGF - FC Midtjylland (2442545),1,away,1000,346.6,351.2,4.6,4,4,"[""186734"", ""551144"", ""420965"", ""494784""]",5.627153,6.85,44.580984,33.090796,15.74,18.25,42.18,-0.18,2.791165,10.104974,1,0,0,0,0,open_play,-2.942727,16.005455,5.387273,13.560000,54.23,28.35,1.912875,53.37,44.66,1.195029,15.336856,11.90,1,2,2.791165,5.504471,10.005224,885.59770,14.785832,15.65,17.894545,5.000000,5.02,0,3.370000,1.865675,0.905388,0.964642,defensive,6
4,4,2024-07-19 AGF - FC Midtjylland (2442545),1,away,1000,435.2,441.0,5.8,3,3,"[""477672"", ""553784"", ""494784""]",6.088612,7.88,49.123461,39.097870,43.53,17.08,1.06,24.85,12.357516,1.426359,1,0,0,0,0,open_play,12.510909,2.595455,24.521818,1.740909,70.98,38.77,1.830797,51.31,63.78,0.804484,26.036112,16.95,0,0,11.187716,12.357516,20.430568,1908.80975,22.095822,16.82,31.304545,13.152946,1.05,0,2.230000,-6.738116,0.134390,0.536112,attacking,7


### Feature Comparison: Successful vs. Failed Pressures

Let's compare the characteristics of successful and failed pressing episodes. We will define 'successful' as `press_success_sustained == 1` and 'failed' as `press_no_effect == 1`. We'll look at a selection of spatial and physical features.

In [ ]:
successful_pressures = press_df_FINAL[press_df_FINAL['press_success_sustained'] == 1].copy()
failed_pressures = press_df_FINAL[press_df_FINAL['press_no_effect'] == 1].copy()

print(f"Number of successful pressures: {len(successful_pressures)}")
print(f"Number of failed pressures: {len(failed_pressures)}")

In [ ]:
comparison_features = [
    "mean_speed", "max_speed", "total_closing", "mean_angle",
    "start_dist_min", "end_dist_min",
    "press_centroid_x", "press_centroid_y",
    "team_length", "team_width", "shape_aspect_ratio",
    "n_pressers_5m", "n_pressers_10m",
    "ball_speed", "nearest_presser_speed", "avg_pressing_team_speed",
    "pressing_intensity", "pressing_intensity_lanes"
]

# Calculate descriptive statistics for successful pressures
success_stats = successful_pressures[comparison_features].agg(['mean', 'std']).T.rename(columns={'mean': 'Successful_Mean', 'std': 'Successful_Std'})

# Calculate descriptive statistics for failed pressures
failed_stats = failed_pressures[comparison_features].agg(['mean', 'std']).T.rename(columns={'mean': 'Failed_Mean', 'std': 'Failed_Std'})

# Combine and display
comparison_df = pd.concat([success_stats, failed_stats], axis=1)
print("\nDescriptive Statistics for Key Features (Successful vs. Failed Pressures):\n")
display(comparison_df.round(2))

### Initial Observations:

Based on the table above, we can observe initial differences:

*   **`mean_speed` and `max_speed`**: Successful pressures tend to have higher mean and max speeds, indicating more intense player movement.
*   **`total_closing`**: Successful pressures show a higher `total_closing`, suggesting players are closing down the opponent more effectively.
*   **`start_dist_min` and `end_dist_min`**: It might be interesting to see if successful pressures start closer to the ball or end closer, or if the *change* in distance is more significant.
*   **`press_centroid_x`**: Differences in `press_centroid_x` could indicate where on the pitch successful pressures typically occur (e.g., more often in the opponent's half).
*   **`n_pressers_5m` / `10m`**: The number of players within certain distances of the ball could highlight the density of the press.
*   **`pressing_intensity` / `pressing_intensity_lanes`**: These are direct measures of pressing effectiveness, and we would expect them to be higher for successful pressures.

To further understand these differences, visualizations like box plots or histograms would be very insightful.